# Stress Prediction V56 — Tie-Aware Pair Neighbor

- 같은 최소 거리에 있는 Train 이웃을 한 명만 고르지 않고 타깃 평균으로 집계
- 8개 핵심 피처의 28개 쌍 예측을 55% 분위수로 결합
- Train-only Leave-One-Out MAE: **0.124299**
- Test 통계 및 Test 학습 사용 없음


In [1]:
# -*- coding: utf-8 -*-
"""Stress Score Prediction V7 — Pair-Neighbor Quantile Blend.

개발 및 검증 환경
- OS: Windows
- Python: 3.12
- numpy: 2.5.1
- pandas: 3.0.5
- scikit-learn: 1.9.0

Train-only 검증으로 고정한 설정
- 독립 V6 Adaptive Feature-Probability ExtraTrees: 85%
- 8개 핵심 피처의 모든 2차원 조합에서 찾은 Train 1-NN 분위수: 15%
- 최종 예측을 0.01 단위로 반올림
- Audit3: V6 0.147019, V7 0.146644, Seed 3/3 승리
- Public MAE: 0.1272333333 (2026-08-01 확인)

Test 통계는 사용하지 않습니다. Test의 각 행은 Train에서 학습한 전처리와
Train 기준 경험적 순위에만 독립적으로 투영됩니다.
"""

from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.55
PAIR_WEIGHT = 1.00

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        minimum_distance = distance.min(axis=1)
        tie_mean = np.array([
            target_values[np.isclose(distance[row], value, rtol=0, atol=1e-8)].mean()
            for row, value in enumerate(minimum_distance)
        ])
        neighbor_targets.append(tie_mean)
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")




In [2]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')
OUTPUT_DIR = Path('/Users/bitsaem/Desktop/stress_project')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
validate_inputs(train, test, submission)
X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)
print('Train:', train.shape, 'Test:', test.shape)


Train: (3000, 18) Test: (3000, 17)


In [3]:
# 최종 학습량과 동일한 2,999 -> 1 Leave-One-Out 검증
train_rank = {}
for column in PAIR_COLUMNS:
    values = X[column].to_numpy(float)
    median = np.nanmedian(values)
    values = np.nan_to_num(values, nan=median)
    sorted_values = np.sort(values)
    train_rank[column] = np.searchsorted(sorted_values, values, side='right') / len(values)

target_values = y.to_numpy(float)
loo_neighbors = []
for first, second in combinations(PAIR_COLUMNS, 2):
    distance = (
        np.abs(train_rank[first][:, None] - train_rank[first][None, :])
        + np.abs(train_rank[second][:, None] - train_rank[second][None, :])
    )
    np.fill_diagonal(distance, np.inf)
    minimum = distance.min(axis=1)
    tie_mean = np.array([
        target_values[np.isclose(distance[row], value, rtol=0, atol=1e-8)].mean()
        for row, value in enumerate(minimum)
    ])
    loo_neighbors.append(tie_mean)
loo_prediction = np.quantile(np.column_stack(loo_neighbors), 0.55, axis=1)
loo_mae = np.mean(np.abs(y.to_numpy() - loo_prediction))
print(f'V56 Leave-One-Out MAE: {loo_mae:.6f}')


V56 Leave-One-Out MAE: 0.124035


In [4]:
prediction = pair_neighbor_prediction(X, X_test, y)
prediction = np.clip(np.round(prediction, 2), 0.0, 1.0)
output = submission.copy()
output[TARGET] = prediction
path = OUTPUT_DIR / 'submit_v56_pair_only_tie_mean_q55.csv'
output.to_csv(path, index=False, encoding='utf-8')
print('Saved:', path)
print('Prediction range:', prediction.min(), prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/submit_v56_pair_only_tie_mean_q55.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.56
4,TEST_0004,0.53
5,TEST_0005,0.66
6,TEST_0006,0.78
7,TEST_0007,0.50
8,TEST_0008,0.57
9,TEST_0009,0.88
